<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/workflow-guides/workflow-01-analyze-apparent-fishing-effort-senegalese-eez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyze apparent fishing effort in Senegalese EEZ

This guide provides detailed instructions to on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to **Analyze apparent fishing effort in [Senegalese EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8371) region and monitor vessel activities** using **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)**, and **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)**.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be easily installed using pip:

In [1]:
# %pip install gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import os

import pandas as pd

import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Introduction

**Use Case: A Port Inspector Monitoring Vessel Activity**

Mamadou, a port inspector in Dakar, Senegal, monitors vessel activity within **[Senegalese Exclusive Economic Zone (EEZ)]((https://www.marineregions.org/gazetteer.php?p=details&id=8371))**. His goal is to:

1. Analyzing apparent fishing effort, specifically for **trawlers** in Senegalese EEZ.
2. Identifying vessels involved in **apparent trawling activity** and determining their reported **flag states**.
3. Checking vessel history, including prior **encounters (or potential transshipment)** or **port visits**.
4. Generating reports for enforcement authorities to assess risks.

**APIs Used:**
️
1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – To retrieve **apparent fishing effort** data for trawlers operating in Senegalese EEZ over the past 3 months.
2. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** – To retrieve **detailed vessel information**, including `flag`, `ownership history`, and `authorizations`.

**Important:** In order to avoid any misinterpretation of **GFW data**, please refer to our official **data caveats** documentations:
- [Apparent fishing effort](https://globalfishingwatch.org/dataset-and-code-fishing-effort/) 
- [Exclusive economic zone boundaries definition](https://globalfishingwatch.org/our-apis/documentation#exclusive-economic-zone-boundaries-definition)
- [Vessel ID](https://globalfishingwatch.org/our-apis/documentation#vessel-id)
- [Vessel API - Vessel identity information](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information)

**Important Caveats:**

1. The [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api) only supports **one active report per user at a time**.
2. **Sending multiple requests simultaneously** results in a **429 Too Many Requests** error.
3. If a report takes over **100 seconds** to generate, it may return a **524 Gateway Timeout** error.

## Step 0: Identify the Region of Interest (ROI) - Senegalese EEZ

Before making API requests, Mamadou must specify the geographic area for analysis using a **Region ID**:

**Options to Define the Region:**

1. **Using Region ID** - Each EEZ has a unique ID in the **[public-eez-areas](https://globalfishingwatch.org/our-apis/documentation#regions)** dataset.
2. **Custom Geometries** - Users can define a custom area using GeoJSON.
   
For **[Senegalese EEZ, the region ID is 8371](https://www.marineregions.org/gazetteer.php?p=details&id=8371)** (public-eez-areas dataset).

## Step 1: Retrieve Apparent Fishing Effort in Senegalese EEZ

Mamadou **first queries** the **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** to get **apparent fishing effort for all vessels**, grouping them by **vessel ID** in **[Senegalese EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8371)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - [8371 Senegalese EEZ]((https://www.marineregions.org/gazetteer.php?p=details&id=8371))
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 3 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Vessel ID
4. **[Gear Type](https://globalfishingwatch.org/our-apis/documentation#gear-types-supported)** - Trawlers 

**Why Use group-by=VESSEL_ID?**

Grouping by **VESSEL_ID** allows **individual vessel identification** in the response. This is crucial for **tracking vessel activity** and, more importantly, linking each detected vessel to the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** in the next step. By structuring the query this way, we can fetch vessel details such as **flag, name, and ownership records** in **Step 2 below**.


**Explanation of Parameters & Considerations**

- Gear types, such as **trawlers**, are inferred based on **Global Fishing Watch’s vessel classification system**, which relies on **AIS data and vessel public registries**. The **gear type associated with each vessel is not always 100% accurate**, as it may be derived from historical sources or inferred from movement patterns. See more details on [supported gear types here](https://globalfishingwatch.org/our-apis/documentation#gear-types-supported).
- Also, please see data caveats regarding [vessel types and their classification here](https://globalfishingwatch.org/our-apis/documentation#vessel-types).
- See more details on retrieving [Region IDs here](https://globalfishingwatch.org/our-apis/documentation#regions).

In [5]:
step_1_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="HIGH",
    group_by="VESSEL_ID",
    temporal_resolution="MONTHLY",
    filters=["geartype in ('trawlers')"],
    start_date="2024-11-01",
    end_date="2025-01-31",
    spatial_aggregation=True,
    region={
        "dataset": "public-eez-areas",
        "id": "8371",
    },
)

In [6]:
step_1_report_df = step_1_report_result.df()

In [7]:
step_1_report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   date                     170 non-null    object             
 1   detections               0 non-null      object             
 2   flag                     170 non-null    object             
 3   gear_type                170 non-null    object             
 4   hours                    170 non-null    float64            
 5   vessel_ids               0 non-null      object             
 6   vessel_id                170 non-null    object             
 7   vessel_type              170 non-null    object             
 8   entry_timestamp          170 non-null    datetime64[ns, UTC]
 9   exit_timestamp           170 non-null    datetime64[ns, UTC]
 10  first_transmission_date  170 non-null    datetime64[ns, UTC]
 11  last_transmission_date   170 non

In [8]:
step_1_report_df[["flag", "gear_type", "hours", "mmsi", "ship_name"]].head()

,flag,gear_type,hours,mmsi,ship_name
0,CHN,TRAWLERS,0.368056,412209175,MENGXIN24
1,ESP,TRAWLERS,1.306389,225987981,CIUDAD DE HUELVA
2,CHN,TRAWLERS,216.545000,412549331,YUAN YU 886
3,SEN,TRAWLERS,612.825556,663123000,ILE AUX OISEAUX
4,CHN,TRAWLERS,31.888889,412444322,MIN LONG YU61146


### Explore Vessels Potentially Engaged in Trawling Activity in the Senegalese EEZ

In [9]:
step_1_agg_report_df = (
    step_1_report_df.groupby(["flag", "gear_type", "mmsi", "ship_name"], as_index=False)
    .agg(hours=("hours", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [10]:
step_1_agg_report_df.head()

,flag,gear_type,mmsi,ship_name,hours
66,SEN,TRAWLERS,663178000,NUEVONOSOLAR,1678.888333
52,SEN,TRAWLERS,663115000,BETTY,1648.771944
49,SEN,TRAWLERS,663112000,TADORNE,1610.828611
42,SEN,TRAWLERS,663039000,SEGUNDO SAN RAFAEL,1595.538333
74,SEN,TRAWLERS,663250000,PRAIA DA MAROSA,1573.587500


### What We have Learned from Step 1

- There are vessels appear to have been engaged in potential trawling activity in Senegalese EEZ over the past 3 months i.e.,:
  - `NUEVONOSOLAR (mmsi: 663178000, flag: SEN)`
  - `BETTY (mmsi: 663115000, flag: SEN)`
- We will retrieve these vessels' `ownership`, `flag history`, and `authorizations` in **Step 2 to validate** them.

## Step 2: Retrieve Vessel Details Using the Vessels API

Mamadou queries the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** to **get detailed vessel identity and ownership records**. Please [learn more about Vessels API here](https://globalfishingwatch.org/our-apis/documentation#vessels-api) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information).

**Filters Used:**

1. **Vessel IDs** from [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api), **Step 1 above**.
2. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)** - `public-global-vessel-identity:latest`.
3. **[Includes](https://globalfishingwatch.org/our-apis/documentation#get-vessels-by-ids-url-parameters)** - `POTENTIAL_RELATED_SELF_REPORTED_INFO`.

**Note:** Vessels may change identifiers over time, such as their `Maritime Mobile Service Identity (MMSI)`,` International Maritime Organization (IMO) number)`, `call sign`, or even their `name`. These changes can occur due to `re-registration`, `changes in ownership`, or other `operational reasons` within the `AIS transponder`. Parameter (`includes = POTENTIAL_RELATED_SELF_REPORTED_INFO`) helps group all **vessel ids** that are **potentially related** as part of the **same physical vessel** based on publicly available registry information.

In [11]:
step_1_vessel_mmsis = list(step_1_agg_report_df["mmsi"].head(n=2))

In [12]:
step_1_vessel_mmsis

['663178000', '663115000']

In [13]:
step_1_vessel_ids = list(
    step_1_report_df[step_1_report_df["mmsi"].isin(step_1_vessel_mmsis)][
        "vessel_id"
    ].unique()
)

In [14]:
step_1_vessel_ids

['894bc3ec6-6ade-f09c-e792-ff2e947508d8',
 'bf28c5a58-8c83-8690-8689-7f2d520f926e']

In [15]:
step_2_vessels_result = await gfw_client.vessels.get_vessels_by_ids(
    ids=step_1_vessel_ids,
)

In [16]:
step_2_vessels_df = step_2_vessels_result.df()

In [17]:
step_2_vessels_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   dataset                         2 non-null      object
 1   registry_info_total_records     2 non-null      int64 
 2   registry_info                   2 non-null      object
 3   registry_owners                 2 non-null      object
 4   registry_public_authorizations  2 non-null      object
 5   combined_sources_info           2 non-null      object
 6   self_reported_info              2 non-null      object
dtypes: int64(1), object(6)
memory usage: 244.0+ bytes


**Understanding Vessel Details Response Data**

- **registryInfoTotalRecords** – This represents the **number of registry records** found for the vessels.
- **registryInfo** – Contains **public registry data**. This data is sourced from official **vessel registries**.
- **registryOwners** – Lists the **registered owners** of the vessel based on public sources.
- **registryPublicAuthorizations** – Represents known **fishing authorizations** from public sources. Users should verify against national registries and RFMO records for additional context.
- **combinedSourcesInfo** – Provides inferred data from multiple sources, including. This is not explicitly reported by vessels but determined through **GFW's classification methods**.
- **selfReportedInfo** – Contains **AIS self-reported** data, including `MMSI`, `ship name`, and `flag` as broadcast by the **vessel itself**. Self-reported data may not always align with registry data and should be cross-checked.

In [18]:
step_2_vessels_df[["registry_info", "registry_owners", "self_reported_info"]]

,registry_info,registry_owners,self_reported_info
0,"[{'id': '29fef17154387858d8d4c777311c57f7', 's...","[{'name': 'SENEVISA', 'flag': 'ESP', 'ssvid': ...",[{'id': 'bf28c5a58-8c83-8690-8689-7f2d520f926e...
1,"[{'id': '199483471cd2da3717552fddb1a3172a', 's...","[{'name': 'ARMEMENT SOPASEN', 'flag': 'SEN', '...",[{'id': '894bc3ec6-6ade-f09c-e792-ff2e947508d8...


### Explore Vessels Registry Info

In [19]:
step_2_registry_info_df = pd.json_normalize(
    step_2_vessels_df["registry_info"].explode()
)

In [20]:
step_2_registry_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "gear_types", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,gear_types,source_code
0,663178000,SEN,NUEVO NOSO LAR,NUEVONOSOLAR,[TRAWLERS],"[IMO, TMT_NATIONAL, TMT_OTHER_OFFICIAL]"
1,762178000,SEN,NUEVO NOSO LAR,NUEVONOSOLAR,[TRAWLERS],"[IMO, TMT_NATIONAL, TMT_OTHER_OFFICIAL]"
2,552178000,SEN,NUEVO NOSO LAR,NUEVONOSOLAR,[TRAWLERS],"[IMO, TMT_NATIONAL, TMT_OTHER_OFFICIAL]"
3,663115000,SEN,BETTY,BETTY,[TRAWLERS],"[IMO, TMT_NATIONAL, TMT_OTHER_OFFICIAL]"


### Explore Registry Owners

In [21]:
step_2_registry_owners_df = pd.json_normalize(
    step_2_vessels_df["registry_owners"].explode()
)

In [22]:
step_2_registry_owners_df[["ssvid", "flag", "name", "source_code"]]

,ssvid,flag,name,source_code
0,663178000,ESP,SENEVISA,"[TMT_NATIONAL, TMT_OTHER_OFFICIAL]"
1,663176000,ESP,SENEVISA,"[TMT_NATIONAL, TMT_OTHER_OFFICIAL]"
2,762178000,ESP,SENEVISA,"[TMT_NATIONAL, TMT_OTHER_OFFICIAL]"
3,552178000,ESP,SENEVISA,"[TMT_NATIONAL, TMT_OTHER_OFFICIAL]"
4,663115000,SEN,ARMEMENT SOPASEN,"[TMT_NATIONAL, TMT_OTHER_OFFICIAL]"


### Explore Vessels Self Reported Info

In [23]:
step_2_self_reported_info_df = pd.json_normalize(
    step_2_vessels_df["self_reported_info"].explode()
)

In [24]:
step_2_self_reported_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,source_code
0,663178000,SEN,NUEVONOSOLAR,NUEVONOSOLAR,[AIS]
1,663178000,SEN,NUEVO=NOSOLAR+3&!U.?,NUEVONOSOLAR3U,[AIS]
2,762178000,None,NUEVO NOSOLAR,NUEVONOSOLAR,[AIS]
3,552178000,None,NUEVO NOSOLAR,NUEVONOSOLAR,[AIS]
4,663178000,SEN,NUEVO NOSOLAR,NUEVONOSOLAR,[AIS]
5,663115000,SEN,BETTY,BETTY,[AIS]


### What We have Learned from Step 2

- **Vessel Identity:**
  - `NUEVONOSOLAR (mmsi: 663178000, flag: SEN)`- appears to be registered under Senegal (SEN)
  - `BETTY (mmsi: 663115000, flag: SEN)` - appears to be registered under Senegal (SEN)
- **Ownership & Historical Changes:**
  - `NUEVONOSOLAR (mmsi: 663178000, flag: SEN)` - **SENEVISA** appears to be listed as the registered owner.
  - `BETTY (mmsi: 663115000, flag: SEN)`- **ARMEMENT SOPASEN** appears to be listed as the registered owner.

**Next Steps:**

- Further, **validate ownership history** using official registry sources.
- Assess whether any **historical changes** in `flag`, `name`, or `ownership` are relevant for enforcement.
- Generate an **apparent activity report** with all available details.

## Summary of API Flow

1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Retrieve apparent fishing effort for **trawlers** within Senegalese EEZ.
2. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** - Fetch detailed **vessel identity**, **ownership history**, and **public authorizations** for vessels detected in **Step 1**.
3. **Analyze vessel history** - Compare **registry records**, **AIS self-reported** data, and inferred information to identify potential flag-hopping or historical changes in vessel identity.
4. **Assess authorizations** - Cross-check whether vessels have publicly available fishing authorizations and consider external official sources for further verification.
5. **Generate an analysis report** - Provide enforcement authorities with a structured report highlighting vessel activity, identity records, and any notable discrepancies for further investigation.